# Perdidas que generan transicion y tiro

Notebook sencillo para construir datasets de modelado de perdidas y transiciones.

La unidad de analisis ya no es una recuperacion aislada. Ahora es:

> una posesion de un equipo que termina y da paso a una posesion rival en juego abierto.

Para cada perdida guardamos:

-   equipo que pierde el balon;
-   equipo que recupera/transiciona;
-   posicion estimada de perdida;
-   accion que explica la perdida (`Block`, `Interception`, `Duel`, `Miscontrol`, etc.);
-   si el rival tira dentro de una ventana configurable;
-   minuto, marcador e igualdad numerica.

In [1]:
# Instala las dependencias necesarias si no están disponibles en el entorno.
# · scipy  viene habitualmente en distribuciones científicas (Anaconda, etc.)
# · shapely es necesaria para el clipping de polígonos Voronoi (sección 8)
import importlib, subprocess, sys

for _pkg in ["scipy", "shapely"]:
    if importlib.util.find_spec(_pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", _pkg])
        print(f"✓ Instalado: {_pkg}")
    else:
        print(f"✓ Ya disponible: {_pkg}")


✓ Instalado: scipy
✓ Instalado: shapely


In [2]:
import warnings

import numpy as np
import pandas as pd
from statsbombpy import sb

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 100)


## 1. Elegir partido

Usamos un partido de Bundesliga 2023/2024. Si quieres otro partido, cambia `MATCH_ID`.

In [3]:
# Bundesliga 2023/2024
COMPETITION_ID = 9
SEASON_ID = 281

# None = usa el primer partido disponible de esa liga/temporada.
# Tambien puedes poner un match_id concreto.
MATCH_ID = None

matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID).copy()
matches = matches.sort_values(["match_date", "match_id"]).reset_index(drop=True)

if MATCH_ID is None:
    MATCH_ID = int(matches.loc[0, "match_id"])

match_info = matches[matches["match_id"] == MATCH_ID].copy()
home_team = match_info.iloc[0]["home_team"]
away_team = match_info.iloc[0]["away_team"]

print(f"MATCH_ID usado: {MATCH_ID}")
display(match_info[["match_id", "match_date", "home_team", "away_team", "home_score", "away_score"]])


MATCH_ID usado: 3895052


,match_id,match_date,home_team,away_team,home_score,away_score
0,3895052,2023-08-19,Bayer Leverkusen,RB Leipzig,3,2


## 2. Cargar eventos

Si falta alguna columna, la creamos con `pd.NA`. Esto evita que el notebook se rompa entre partidos con columnas distintas.

In [4]:
COLS_NEEDED = [
    "id", "index", "period", "timestamp", "minute", "second",
    "type", "play_pattern", "possession", "possession_team", "team", "player",
    "location", "out", "under_pressure", "counterpress",
    "shot_outcome", "shot_statsbomb_xg",
    "ball_recovery_recovery_failure",
    "ball_recovery_offensive",
    "block_deflection", "block_offensive", "block_save_block",
    "interception_outcome",
    "duel_type", "duel_outcome",
    "50_50_outcome",
    "clearance_aerial_won", "clearance_body_part",
    "pass_outcome", "pass_end_location", "dribble_outcome", "ball_receipt_outcome",
    "foul_committed_card", "bad_behaviour_card",
]


def has_xy(value):
    return isinstance(value, (list, tuple)) and len(value) >= 2


def get_x(value):
    return float(value[0]) if has_xy(value) else np.nan


def get_y(value):
    return float(value[1]) if has_xy(value) else np.nan


def get_end_x(value):
    return float(value[0]) if has_xy(value) else np.nan


def get_end_y(value):
    return float(value[1]) if has_xy(value) else np.nan


def prepare_events(raw_events):
    events = raw_events.copy()

    # Rellenamos columnas que no existan en este partido.
    for col in COLS_NEEDED:
        if col not in events.columns:
            events[col] = pd.NA

    events = events[COLS_NEEDED].copy()

    # Orden estable de eventos dentro del partido.
    fallback_order = pd.Series(np.arange(len(events)), index=events.index)
    events["event_order"] = pd.to_numeric(events["index"], errors="coerce").fillna(fallback_order)

    # Coordenadas StatsBomb: location = [x, y].
    events["x"] = events["location"].apply(get_x)
    events["y"] = events["location"].apply(get_y)
    events["pass_end_x"] = events["pass_end_location"].apply(get_end_x)
    events["pass_end_y"] = events["pass_end_location"].apply(get_end_y)

    events["period"] = pd.to_numeric(events["period"], errors="coerce").fillna(0).astype(int)
    events["minute"] = pd.to_numeric(events["minute"], errors="coerce")
    events["second"] = pd.to_numeric(events["second"], errors="coerce")
    events["event_seconds"] = events["minute"].fillna(0) * 60 + events["second"].fillna(0)
    events["possession"] = pd.to_numeric(events["possession"], errors="coerce")
    events["match_id"] = int(MATCH_ID)

    return events.sort_values(["period", "event_order"]).reset_index(drop=True)


raw_events = sb.events(match_id=MATCH_ID)
events = prepare_events(raw_events)

# Datos 360: archivo complementario con fotogramas de posiciones de jugadores.
# statsbombpy los expone a través de sb.frames(), que devuelve UNA FILA POR JUGADOR
# y renombra event_uuid → id. Reagrupamos por id para reconstruir la lista de
# jugadores por evento, que es el formato que necesita zone_variables().
try:
    _frames = sb.frames(match_id=MATCH_ID)
    freeze_lookup = {
        event_id: grp[["teammate", "actor", "keeper", "location"]].to_dict("records")
        for event_id, grp in _frames.groupby("id")
    }
    print(f"Eventos con fotograma 360: {len(freeze_lookup):,}")
except Exception as _exc:
    print(f"⚠ Sin datos 360 para este partido ({_exc}). Variables de zona → NaN.")
    freeze_lookup = {}

print(f"Eventos cargados: {len(events):,}")
events.head()


⚠ Sin datos 360 para este partido (Reindexing only valid with uniquely valued Index objects). Variables de zona → NaN.
Eventos cargados: 3,739


,id,index,period,timestamp,minute,second,type,play_pattern,possession,possession_team,team,player,location,out,under_pressure,counterpress,shot_outcome,shot_statsbomb_xg,ball_recovery_recovery_failure,ball_recovery_offensive,block_deflection,block_offensive,block_save_block,interception_outcome,duel_type,duel_outcome,50_50_outcome,clearance_aerial_won,clearance_body_part,pass_outcome,pass_end_location,dribble_outcome,ball_receipt_outcome,foul_committed_card,bad_behaviour_card,event_order,x,y,pass_end_x,pass_end_y,event_seconds,match_id
0,69bb4fa7-f177-49a8-9b19-7c01cd9d9a9c,1,1,00:00:00.000,0,0,Starting XI,Regular Play,1,Bayer Leverkusen,Bayer Leverkusen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,0,3895052
1,9790a466-9c56-42aa-96aa-a9ad1a6f8811,2,1,00:00:00.000,0,0,Starting XI,Regular Play,1,Bayer Leverkusen,RB Leipzig,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,0,3895052
2,c6762724-3ad0-4ed1-b232-7f32ac192ec6,3,1,00:00:00.000,0,0,Half Start,Regular Play,1,Bayer Leverkusen,Bayer Leverkusen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN,NaN,0,3895052
3,ba22283a-c920-448c-804e-3737cccd9fff,4,1,00:00:00.000,0,0,Half Start,Regular Play,1,Bayer Leverkusen,RB Leipzig,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN,NaN,NaN,0,3895052
4,9a50295b-7f7a-449c-9ef3-87e60467e8e1,5,1,00:00:00.773,0,0,Pass,From Kick Off,2,RB Leipzig,RB Leipzig,Ikoma Loïs Openda,"[61.0, 40.1]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,"[57.0, 40.8]",NaN,NaN,NaN,NaN,5,61.0,40.1,57.0,40.8,0,3895052


In [20]:
events[(events["possession"] == 43) | (events["possession"] == 44)]

,id,index,period,timestamp,minute,second,type,play_pattern,possession,possession_team,team,player,location,out,under_pressure,counterpress,shot_outcome,shot_statsbomb_xg,ball_recovery_recovery_failure,ball_recovery_offensive,block_deflection,block_offensive,block_save_block,interception_outcome,duel_type,duel_outcome,50_50_outcome,clearance_aerial_won,clearance_body_part,pass_outcome,pass_end_location,dribble_outcome,ball_receipt_outcome,foul_committed_card,bad_behaviour_card,event_order,x,y,pass_end_x,pass_end_y,event_seconds,match_id
1061,4dd53f83-48c6-4df0-9892-ad521a689c5d,1062,1,00:22:39.085,22,39,Ball Recovery,Regular Play,43,RB Leipzig,RB Leipzig,Daniel Olmo Carvajal,"[41.6, 39.6]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1062,41.6,39.6,NaN,NaN,1359,3895052
1062,5697fe89-6311-4a29-9ed5-29a4e0579662,1063,1,00:22:39.085,22,39,Carry,Regular Play,43,RB Leipzig,RB Leipzig,Daniel Olmo Carvajal,"[41.6, 39.6]",NaN,True,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1063,41.6,39.6,NaN,NaN,1359,3895052
1063,c64dc864-9309-4a3c-b97d-3ed6b119a7c9,1064,1,00:22:39.611,22,39,Pressure,Regular Play,43,RB Leipzig,Bayer Leverkusen,Exequiel Alejandro Palacios,"[77.3, 39.3]",NaN,NaN,True,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1064,77.3,39.3,NaN,NaN,1359,3895052
1064,f78e1d88-e6c8-43d9-8727-50f8216532d2,1065,1,00:22:42.627,22,42,Pass,Regular Play,43,RB Leipzig,RB Leipzig,Daniel Olmo Carvajal,"[47.4, 56.9]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,"[38.6, 71.3]",NaN,NaN,NaN,NaN,1065,47.4,56.9,38.6,71.3,1362,3895052
1065,3cc03df9-09de-4770-9bde-3dc3e1be06bf,1066,1,00:22:44.746,22,44,Ball Receipt*,Regular Play,43,RB Leipzig,RB Leipzig,Benjamin Henrichs,"[38.6, 71.3]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1066,38.6,71.3,NaN,NaN,1364,3895052
1066,144969ee-b234-4a3f-9a20-780320568d54,1067,1,00:22:44.746,22,44,Carry,Regular Play,43,RB Leipzig,RB Leipzig,Benjamin Henrichs,"[38.6, 71.3]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1067,38.6,71.3,NaN,NaN,1364,3895052
1067,11c59ad4-dcc3-4d02-87e7-ed6fc15407c1,1068,1,00:22:44.942,22,44,Pass,Regular Play,43,RB Leipzig,RB Leipzig,Benjamin Henrichs,"[38.4, 71.7]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,"[29.1, 63.1]",NaN,NaN,NaN,NaN,1068,38.4,71.7,29.1,63.1,1364,3895052
1068,91e07e83-7e16-48c2-a78d-b5442072b48f,1069,1,00:22:46.010,22,46,Ball Receipt*,Regular Play,43,RB Leipzig,RB Leipzig,Mohamed Simakan,"[29.1, 63.1]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1069,29.1,63.1,NaN,NaN,1366,3895052
1069,7f72c850-78f8-4f2a-a238-c4b017cd12a5,1070,1,00:22:46.010,22,46,Carry,Regular Play,43,RB Leipzig,RB Leipzig,Mohamed Simakan,"[29.1, 63.1]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1070,29.1,63.1,NaN,NaN,1366,3895052
1070,c0a11d57-e4a5-4ee9-9ddd-a31521a34f8e,1071,1,00:22:47.449,22,47,Pass,Regular Play,43,RB Leipzig,RB Leipzig,Mohamed Simakan,"[24.9, 57.9]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,"[12.0, 52.9]",NaN,NaN,NaN,NaN,1071,24.9,57.9,12.0,52.9,1367,3895052


## 3. Concepto: perdida vs recuperacion

Una **perdida** se mira desde el equipo que tenia la posesion.

Una **recuperacion** se mira desde el rival que roba o recoge el balon.

Para la regresion queremos una fila por perdida:

``` text
equipo A ataca -> equipo A pierde -> equipo B transiciona -> equipo B tira o no tira dentro de la ventana
```

Por eso partimos de cambios de posesion entre equipos, no de eventos sueltos.

In [14]:
OPEN_PLAY_NEXT_PATTERNS = {"Regular Play", "From Counter"}
SHOT_WINDOW_SECONDS = 10


def is_true(value):
    if pd.isna(value):
        return False
    if isinstance(value, str):
        return value.lower() in {"true", "1", "yes", "si", "sí"}
    return bool(value)


def possession_meta(events):
    """Una fila por posesion: equipo, inicio y final."""
    return (
        events.dropna(subset=["possession"])
        .sort_values(["period", "event_order"])
        .groupby(["match_id", "period", "possession"], observed=True)
        .agg(
            possession_team=("possession_team", "first"),
            play_pattern=("play_pattern", "first"),
            start_order=("event_order", "min"),
            end_order=("event_order", "max"),
            start_seconds=("event_seconds", "min"),
            end_seconds=("event_seconds", "max"),
        )
        .reset_index()
        .sort_values(["period", "start_order"])
        .reset_index(drop=True)
    )


meta = possession_meta(events)
meta.head(100)


,match_id,period,possession,possession_team,play_pattern,start_order,end_order,start_seconds,end_seconds
0,3895052,1,1,Bayer Leverkusen,Regular Play,1,4,0,0
1,3895052,1,2,RB Leipzig,From Kick Off,5,38,0,29
2,3895052,1,3,RB Leipzig,From Throw In,39,57,46,59
3,3895052,1,4,RB Leipzig,From Free Kick,58,87,72,94
4,3895052,1,5,RB Leipzig,From Throw In,88,95,105,110
5,3895052,1,6,Bayer Leverkusen,Regular Play,96,111,112,119
6,3895052,1,7,RB Leipzig,Regular Play,112,112,120,120
7,3895052,1,8,RB Leipzig,From Keeper,113,117,127,137
8,3895052,1,9,Bayer Leverkusen,Regular Play,118,130,137,146
9,3895052,1,10,RB Leipzig,Regular Play,131,144,147,157


## 4. Elegir la accion que explica la perdida

Para cada cambio de posesion en juego abierto:

1.  `losing_team` es el `possession_team` de la posesion que acaba.
2.  `transition_team` es el `possession_team` de la siguiente posesion.
3.  Buscamos cerca del cambio una accion que explique la perdida.

Priorizamos acciones defensivas del equipo que roba: `Block`, `Interception`, `Duel/Tackle`, `50/50`, `Clearance`.

Si no hay una accion defensiva dentro de la posesion que acaba, aceptamos un `Ball Recovery` solo cuando es el primer evento de la nueva posesion y la posesion anterior contiene una señal clara de perdida (`Ball Receipt* Incomplete`, `Miscontrol`, `Dispossessed`, `Pass Incomplete`, etc.). Asi capturamos balones sueltos sin usar ventanas arbitrarias.

Si no encontramos accion defensiva clara, usamos la ultima accion de ataque del equipo que pierde: `Miscontrol`, `Dispossessed`, `Dribble`, `Pass`, etc.

In [7]:
DEFENSIVE_PRIORITY = {
    "Block": 1,
    "Interception": 2,
    "Duel": 3,
    "50/50": 4,
    "Clearance": 5,
}

ATTACKING_PRIORITY = {
    "Miscontrol": 1,
    "Dispossessed": 2,
    "Dribble": 3,
    "Pass": 4,
    "Carry": 5,
    "Ball Receipt*": 6,
}

VALID_TACKLE_OUTCOMES = {"Won", "Success", "Success In Play"}
VALID_INTERCEPTION_OUTCOMES = {"Won", "Success", "In Play", "Success In Play"}
VALID_50_50_OUTCOMES = {"Won", "Success To Team"}
INCOMPLETE_PASS_OUTCOMES = {"Incomplete", "Out", "Pass Offside", "Unknown", "Injury Clearance"}
INCOMPLETE_DRIBBLE_OUTCOMES = {"Incomplete"}
INCOMPLETE_RECEIPT_OUTCOMES = {"Incomplete"}
SHOT_END_TYPES = {"Shot", "Goal Keeper"}


def mirror_x(x):
    return PITCH_LENGTH - x if pd.notna(x) else np.nan


def mirror_y(y):
    return PITCH_WIDTH - y if pd.notna(y) else np.nan


PITCH_LENGTH = 120
PITCH_WIDTH = 80


def is_restart_or_out(previous_events, next_possession):
    """Excluye cambios por banda, faltas, corners, saques, etc."""
    if next_possession["play_pattern"] not in OPEN_PLAY_NEXT_PATTERNS:
        return True

    if previous_events.empty:
        return True

    last_event = previous_events.sort_values("event_order").iloc[-1]

    # Si la posesion anterior contiene un tiro, no es una perdida de las que
    # queremos modelar. Es una posesion que ya termino atacando.
    if (previous_events["type"] == "Shot").any():
        return True

    # Si acaba con accion de portero, tambien suele ser una continuacion natural
    # tras tiro, parada, recepcion del portero o reinicio.
    if str(last_event.get("type", "")) == "Goal Keeper":
        return True

    if is_true(last_event.get("out", pd.NA)):
        return True

    if str(last_event.get("pass_outcome", "")) in {"Out", "Pass Offside", "Injury Clearance"}:
        return True

    return False


def has_attacking_loss_signal(event):
    """True si el evento del equipo que ataca describe perdida o mal control."""
    event_type = event.get("type")

    if event_type in {"Miscontrol", "Dispossessed"}:
        return True
    if event_type == "Ball Receipt*":
        return str(event.get("ball_receipt_outcome")) in INCOMPLETE_RECEIPT_OUTCOMES
    if event_type == "Dribble":
        return str(event.get("dribble_outcome")) in INCOMPLETE_DRIBBLE_OUTCOMES
    if event_type == "Pass":
        return str(event.get("pass_outcome")) in INCOMPLETE_PASS_OUTCOMES
    if event_type == "Duel":
        return "Lost" in str(event.get("duel_outcome"))
    return False


def attacking_loss_candidates(previous_events, losing_team):
    """Eventos del equipo que pierde que explican la perdida."""
    candidates = previous_events[
        (previous_events["team"] == losing_team)
        & (previous_events["type"].isin(ATTACKING_PRIORITY) | (previous_events["type"] == "Duel"))
        & previous_events["x"].notna()
        & previous_events["y"].notna()
    ].copy()

    if candidates.empty:
        return candidates

    candidates = candidates[candidates.apply(has_attacking_loss_signal, axis=1)].copy()
    if candidates.empty:
        return candidates

    candidates["priority"] = candidates["type"].map(ATTACKING_PRIORITY).fillna(7)
    return candidates


def first_next_possession_recovery(events, next_possession, transition_team):
    """Ball Recovery valido solo si abre la nueva posesion."""
    next_events = events[
        (events["period"] == int(next_possession["period"]))
        & (events["possession"] == next_possession["possession"])
    ].sort_values("event_order").copy()

    if next_events.empty:
        return pd.Series(dtype="object")

    first_event = next_events.iloc[0]
    if (
        first_event.get("type") == "Ball Recovery"
        and first_event.get("team") == transition_team
        and pd.isna(first_event.get("ball_recovery_recovery_failure"))
        and pd.notna(first_event.get("x"))
        and pd.notna(first_event.get("y"))
    ):
        return first_event

    return pd.Series(dtype="object")


def event_location_for_loss(event):
    """Punto fisico de perdida para el evento elegido.

    Para un pase incompleto, preferimos el destino del pase si existe.
    Para el resto usamos la localizacion del evento.
    """
    if event.get("type") == "Pass" and pd.notna(event.get("pass_end_x")) and pd.notna(event.get("pass_end_y")):
        return float(event["pass_end_x"]), float(event["pass_end_y"])
    return float(event["x"]), float(event["y"])


def filter_valid_defensive_candidates(candidates):
    """Evita meter duelos/intercepciones que no ganan realmente la pelota."""
    if candidates.empty:
        return candidates

    valid = []
    for _, event in candidates.iterrows():
        event_type = event["type"]

        if event_type == "Duel":
            valid.append(
                event.get("duel_type") == "Tackle"
                and str(event.get("duel_outcome")) in VALID_TACKLE_OUTCOMES
            )
        elif event_type == "Interception":
            valid.append(
                str(event.get("interception_outcome")) in VALID_INTERCEPTION_OUTCOMES
                or pd.isna(event.get("interception_outcome"))
            )
        elif event_type == "50/50":
            valid.append(str(event.get("50_50_outcome")) in VALID_50_50_OUTCOMES)
        else:
            valid.append(True)

    return candidates[pd.Series(valid, index=candidates.index)].copy()


def choose_turnover_event(events, current_possession, next_possession):
    """Elige el evento que mejor representa la perdida."""
    losing_team = current_possession["possession_team"]
    transition_team = next_possession["possession_team"]
    period = int(current_possession["period"])

    # Eventos de la posesion que acaba.
    previous_events = events[
        (events["period"] == period)
        & (events["possession"] == current_possession["possession"])
    ].copy()

    # Primero buscamos acciones defensivas del equipo que transiciona, pero
    # SOLO dentro de la posesion que pierde el rival.
    defensive_candidates = previous_events.copy()
    defensive_candidates = defensive_candidates[
        (defensive_candidates["team"] == transition_team)
        & (defensive_candidates["type"].isin(DEFENSIVE_PRIORITY))
        & defensive_candidates["x"].notna()
        & defensive_candidates["y"].notna()
    ].copy()
    defensive_candidates = filter_valid_defensive_candidates(defensive_candidates)

    if not defensive_candidates.empty:
        defensive_candidates["priority"] = defensive_candidates["type"].map(DEFENSIVE_PRIORITY)
        event = defensive_candidates.sort_values(["priority", "event_order"]).iloc[0]
        source = "defensive_event"
    else:
        attacking_candidates = attacking_loss_candidates(previous_events, losing_team)

        # Ball Recovery entra solo si la posesion previa tiene una perdida clara
        # y el Ball Recovery abre la nueva posesion.
        next_recovery = first_next_possession_recovery(events, next_possession, transition_team)
        if not attacking_candidates.empty and not next_recovery.empty:
            event = next_recovery
            source = "next_possession_recovery"
        elif not attacking_candidates.empty:
            # Si no hay recuperacion inicial, usamos la ultima señal de perdida
            # del equipo que atacaba.
            event = attacking_candidates.sort_values(["event_order", "priority"], ascending=[False, True]).iloc[0]
            source = "attacking_event"
        else:
            return None, previous_events

    event_x, event_y = event_location_for_loss(event)

    return {
        "turnover_event_source": source,
        "turnover_event_id": event.get("id", pd.NA),
        "turnover_action_type": event.get("type", pd.NA),
        "turnover_player": event.get("player", pd.NA),
        "turnover_event_team": event.get("team", pd.NA),
        "turnover_event_order": float(event["event_order"]),
        "turnover_event_seconds": float(event["event_seconds"]),
        "turnover_x": event_x,
        "turnover_y": event_y,
        "raw_event_x": float(event["x"]),
        "raw_event_y": float(event["y"]),
        "pass_outcome": event.get("pass_outcome", pd.NA),
        "dribble_outcome": event.get("dribble_outcome", pd.NA),
        "ball_receipt_outcome": event.get("ball_receipt_outcome", pd.NA),
        "duel_type": event.get("duel_type", pd.NA),
        "duel_outcome": event.get("duel_outcome", pd.NA),
        "interception_outcome": event.get("interception_outcome", pd.NA),
        "50_50_outcome": event.get("50_50_outcome", pd.NA),
        "clearance_aerial_won": event.get("clearance_aerial_won", pd.NA),
        "clearance_body_part": event.get("clearance_body_part", pd.NA),
        "ball_recovery_offensive": event.get("ball_recovery_offensive", pd.NA),
        "block_deflection": event.get("block_deflection", pd.NA),
        "block_offensive": event.get("block_offensive", pd.NA),
        "block_save_block": event.get("block_save_block", pd.NA),
        "under_pressure": event.get("under_pressure", pd.NA),
        "counterpress": event.get("counterpress", pd.NA),
    }, previous_events


## 5. Construir dataset de perdidas

Guardamos dos pares de coordenadas:

-   `loss_x`, `loss_y`: punto visto desde el equipo que pierde el balon. Son las variables principales para modelar riesgo de perdida.
-   `transition_x`, `transition_y`: el mismo punto visto desde el equipo que roba/transiciona. Sirven para debug y mapas desde el equipo que ataca despues.

Si el evento elegido lo hace el rival (`defensive_event` o `next_possession_recovery`), `turnover_x/y` ya estan en perspectiva del equipo que transiciona y los espejamos para obtener `loss_x/y`.

Si el evento elegido lo hace el equipo que pierde (`attacking_event`), `turnover_x/y` ya son `loss_x/y` y los espejamos para obtener `transition_x/y`.

In [8]:
def first_shot_after_turnover(events, transition_team, next_possession, t0_seconds, window_seconds=10):
    """Primer tiro del equipo que transiciona en su siguiente posesion y dentro de la ventana."""
    shots = events[
        (events["period"] == int(next_possession["period"]))
        & (events["possession"] == next_possession["possession"])
        & (events["team"] == transition_team)
        & (events["type"] == "Shot")
    ].copy()

    if shots.empty:
        return {
            "transition_shot_within_window": 0,
            "seconds_to_shot": np.nan,
            "shot_player": pd.NA,
            "shot_outcome": pd.NA,
            "shot_statsbomb_xg": np.nan,
        }

    shots["seconds_to_shot"] = shots["event_seconds"] - t0_seconds
    shots = shots[
        (shots["seconds_to_shot"] >= 0)
        & (shots["seconds_to_shot"] <= window_seconds)
    ].sort_values(["seconds_to_shot", "event_order"])

    if shots.empty:
        return {
            "transition_shot_within_window": 0,
            "seconds_to_shot": np.nan,
            "shot_player": pd.NA,
            "shot_outcome": pd.NA,
            "shot_statsbomb_xg": np.nan,
        }

    shot = shots.iloc[0]
    return {
        "transition_shot_within_window": 1,
        "seconds_to_shot": float(shot["seconds_to_shot"]),
        "shot_player": shot.get("player", pd.NA),
        "shot_outcome": shot.get("shot_outcome", pd.NA),
        "shot_statsbomb_xg": shot.get("shot_statsbomb_xg", np.nan),
    }


rows = []

for i in range(len(meta) - 1):
    current_poss = meta.iloc[i]
    next_poss = meta.iloc[i + 1]

    # Solo nos interesan cambios de posesion dentro del mismo periodo.
    if current_poss["period"] != next_poss["period"]:
        continue

    losing_team = current_poss["possession_team"]
    transition_team = next_poss["possession_team"]

    # Si sigue el mismo equipo, no hay perdida entre equipos.
    if losing_team == transition_team:
        continue

    turnover_event, previous_events = choose_turnover_event(events, current_poss, next_poss)
    if turnover_event is None:
        continue

    # Excluimos cambios por banda/falta/reinicio. Queremos transiciones en juego.
    if is_restart_or_out(previous_events, next_poss):
        continue

    if turnover_event["turnover_event_source"] in {"defensive_event", "next_possession_recovery"}:
        loss_x = mirror_x(turnover_event["turnover_x"])
        loss_y = mirror_y(turnover_event["turnover_y"])
        transition_x = turnover_event["turnover_x"]
        transition_y = turnover_event["turnover_y"]
    else:
        loss_x = turnover_event["turnover_x"]
        loss_y = turnover_event["turnover_y"]
        transition_x = mirror_x(turnover_event["turnover_x"])
        transition_y = mirror_y(turnover_event["turnover_y"])

    shot_info = first_shot_after_turnover(
        events=events,
        transition_team=transition_team,
        next_possession=next_poss,
        # La transicion empieza cuando arranca la nueva posesion del rival,
        # no necesariamente en el evento defensivo que causo la perdida.
        t0_seconds=float(next_poss["start_seconds"]),
        window_seconds=SHOT_WINDOW_SECONDS,
    )

    rows.append({
        "match_id": int(current_poss["match_id"]),
        "period": int(current_poss["period"]),
        "losing_team": losing_team,
        "transition_team": transition_team,
        "lost_possession": current_poss["possession"],
        "transition_possession": next_poss["possession"],
        "lost_possession_play_pattern": current_poss["play_pattern"],
        "transition_play_pattern": next_poss["play_pattern"],
        "action_minute": round(float(next_poss["start_seconds"]) / 60, 2),
        "event_seconds": float(next_poss["start_seconds"]),
        "loss_x": loss_x,
        "loss_y": loss_y,
        "transition_x": transition_x,
        "transition_y": transition_y,
        **turnover_event,
        **shot_info,
    })

loss_transition_events = pd.DataFrame(rows)

print(f"Perdidas en juego detectadas: {len(loss_transition_events):,}")
print(f"Perdidas con tiro rival en <= {SHOT_WINDOW_SECONDS}s: {int(loss_transition_events['transition_shot_within_window'].sum()) if not loss_transition_events.empty else 0:,}")
loss_transition_events.head(20)


Perdidas en juego detectadas: 70
Perdidas con tiro rival en <= 10s: 3


,match_id,period,losing_team,transition_team,lost_possession,transition_possession,lost_possession_play_pattern,transition_play_pattern,action_minute,event_seconds,loss_x,loss_y,transition_x,transition_y,turnover_event_source,turnover_event_id,turnover_action_type,turnover_player,turnover_event_team,turnover_event_order,turnover_event_seconds,turnover_x,turnover_y,raw_event_x,raw_event_y,pass_outcome,dribble_outcome,ball_receipt_outcome,duel_type,duel_outcome,interception_outcome,50_50_outcome,clearance_aerial_won,clearance_body_part,ball_recovery_offensive,block_deflection,block_offensive,block_save_block,under_pressure,counterpress,transition_shot_within_window,seconds_to_shot,shot_player,shot_outcome,shot_statsbomb_xg
0,3895052,1,RB Leipzig,Bayer Leverkusen,5,6,From Throw In,Regular Play,1.87,112.0,19.6,24.2,100.4,55.8,defensive_event,c3323881-9112-4b7d-b3cf-06062fcb5902,Block,Granit Xhaka,Bayer Leverkusen,95.0,110.0,100.4,55.8,100.4,55.8,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,1,7.0,Granit Xhaka,Saved,0.020734
1,3895052,1,RB Leipzig,Bayer Leverkusen,8,9,From Keeper,Regular Play,2.28,137.0,35.1,18.3,84.9,61.7,attacking_event,d684e5e4-ee5a-4a7a-8013-d787ae77e932,Miscontrol,Xaver Schlager,RB Leipzig,117.0,137.0,35.1,18.3,35.1,18.3,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN
2,3895052,1,Bayer Leverkusen,RB Leipzig,9,10,Regular Play,Regular Play,2.45,147.0,97.9,67.1,22.1,12.9,defensive_event,5bc87cbd-db36-4bd5-a4d7-c07ed5435238,Clearance,Daniel Olmo Carvajal,RB Leipzig,130.0,146.0,22.1,12.9,22.1,12.9,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,True,Head,NaN,<NA>,<NA>,<NA>,True,NaN,0,NaN,NaN,NaN,NaN
3,3895052,1,RB Leipzig,Bayer Leverkusen,11,12,From Throw In,Regular Play,3.13,188.0,19.0,11.8,101.0,68.2,defensive_event,bbf1e609-60cb-4dcb-bd23-9ac34721002d,Duel,Jonas Hofmann,Bayer Leverkusen,150.0,181.0,101.0,68.2,101.0,68.2,NaN,NaN,NaN,Tackle,Success In Play,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,<NA>,True,NaN,0,NaN,NaN,NaN,NaN
4,3895052,1,Bayer Leverkusen,RB Leipzig,12,13,Regular Play,Regular Play,4.13,248.0,108.1,55.6,11.9,24.4,attacking_event,5bc6657b-a17c-486e-ad59-22672d97b128,Ball Receipt*,Jeremie Frimpong,Bayer Leverkusen,219.0,248.0,108.1,55.6,108.1,55.6,NaN,NaN,Incomplete,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN
5,3895052,1,RB Leipzig,Bayer Leverkusen,14,15,From Free Kick,Regular Play,4.97,298.0,63.5,16.3,56.5,63.7,attacking_event,da6e321b-2781-4b97-b334-1e563ecebe3f,Ball Receipt*,Daniel Olmo Carvajal,RB Leipzig,255.0,298.0,63.5,16.3,63.5,16.3,NaN,NaN,Incomplete,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN
6,3895052,1,RB Leipzig,Bayer Leverkusen,16,17,From Throw In,Regular Play,5.65,339.0,116.9,56.3,3.1,23.7,next_possession_recovery,f7a5cf2b-d965-4472-9fb4-c60e51e5c022,Ball Recovery,Lukáš Hrádecký,Bayer Leverkusen,300.0,339.0,3.1,23.7,3.1,23.7,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN
7,3895052,1,Bayer Leverkusen,RB Leipzig,17,18,Regular Play,Regular Play,6.83,410.0,89.5,52.8,30.5,27.2,defensive_event,bc3e5394-92ad-4eb2-bcf9-454d8336659c,Duel,Xaver Schlager,RB Leipzig,347.0,383.0,30.5,27.2,30.5,27.2,NaN,NaN,NaN,Tackle,Success In Play,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,<NA>,True,NaN,0,NaN,NaN,NaN,NaN
8,3895052,1,RB Leipzig,Bayer Leverkusen,25,26,From Throw In,Regular Play,10.87,652.0,54.1,16.6,65.9,63.4,next_possession_recovery,0850ac23-450e-428c-923b-76bcb9c765f3,Ball Recovery,Granit Xhaka,Bayer Leverkusen,551.0,652.0,65.9,63.4,65.9,63.4,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN
9,3895052,1,RB Leipzig,Bayer Leverkusen,27,28,From Goal Kick,Regular Play,12.32,739.0,51.8,21.6,68.2,58.4,attacking_event,e0775f4b-63cc-43b8-9eec-e82403cae033,Ball Receipt*,Timo Werner,RB Leipzig,612.0,739.0,51.8,21.6,51.8,21.6,NaN,NaN,Incomplete,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN


## 6. Contexto de marcador e igualdad numerica

El contexto se calcula desde el punto de vista del equipo que pierde el balon.

In [9]:
def opponent_of(team):
    if team == home_team:
        return away_team
    if team == away_team:
        return home_team
    return pd.NA


goals = (
    events[(events["type"] == "Shot") & (events["shot_outcome"] == "Goal")]
    [["period", "event_order", "event_seconds", "team"]]
    .sort_values(["period", "event_order"])
    .reset_index(drop=True)
)

red_card_values = {"Red Card", "Second Yellow", "Second Yellow Card"}
red_cards = events[
    events[["foul_committed_card", "bad_behaviour_card"]]
    .apply(lambda row: any(str(value) in red_card_values for value in row.dropna()), axis=1)
][["period", "event_order", "event_seconds", "team"]].copy()


def score_state_before(period, event_order, team):
    """Estado de marcador de un equipo antes de una accion."""
    rival = opponent_of(team)
    previous_goals = goals[
        (goals["period"] < period)
        | ((goals["period"] == period) & (goals["event_order"] < event_order))
    ]

    team_goals = int((previous_goals["team"] == team).sum())
    rival_goals = int((previous_goals["team"] == rival).sum())
    diff = team_goals - rival_goals

    if diff > 0:
        return "winning", diff
    if diff < 0:
        return "losing", diff
    return "drawing", diff


def numerical_equality_before(period, event_order, team):
    """Igualdad numerica antes de una accion, restando rojas previas."""
    rival = opponent_of(team)
    previous_reds = red_cards[
        (red_cards["period"] < period)
        | ((red_cards["period"] == period) & (red_cards["event_order"] < event_order))
    ]

    team_players = 11 - int((previous_reds["team"] == team).sum())
    rival_players = 11 - int((previous_reds["team"] == rival).sum())
    return team_players == rival_players, team_players, rival_players


context_rows = []
for _, loss in loss_transition_events.iterrows():
    score_state, score_diff = score_state_before(
        period=int(loss["period"]),
        event_order=float(loss["turnover_event_order"]),
        team=loss["losing_team"],
    )
    is_equal, losing_players, transition_players = numerical_equality_before(
        period=int(loss["period"]),
        event_order=float(loss["turnover_event_order"]),
        team=loss["losing_team"],
    )

    context_rows.append({
        "losing_team_score_state": score_state,
        "losing_team_score_diff": score_diff,
        "losing_team_players": losing_players,
        "transition_team_players": transition_players,
        # Positivo: el equipo que roba/transiciona tiene mas jugadores.
        # Negativo: el equipo que roba/transiciona tiene menos jugadores.
        "transition_team_player_diff": transition_players - losing_players,
        "is_numerical_equality": is_equal,
    })

context = pd.DataFrame(context_rows)
model_dataset = pd.concat([loss_transition_events.reset_index(drop=True), context], axis=1)

model_dataset.head()


,match_id,period,losing_team,transition_team,lost_possession,transition_possession,lost_possession_play_pattern,transition_play_pattern,action_minute,event_seconds,loss_x,loss_y,transition_x,transition_y,turnover_event_source,turnover_event_id,turnover_action_type,turnover_player,turnover_event_team,turnover_event_order,turnover_event_seconds,turnover_x,turnover_y,raw_event_x,raw_event_y,pass_outcome,dribble_outcome,ball_receipt_outcome,duel_type,duel_outcome,interception_outcome,50_50_outcome,clearance_aerial_won,clearance_body_part,ball_recovery_offensive,block_deflection,block_offensive,block_save_block,under_pressure,counterpress,transition_shot_within_window,seconds_to_shot,shot_player,shot_outcome,shot_statsbomb_xg,losing_team_score_state,losing_team_score_diff,losing_team_players,transition_team_players,transition_team_player_diff,is_numerical_equality
0,3895052,1,RB Leipzig,Bayer Leverkusen,5,6,From Throw In,Regular Play,1.87,112.0,19.6,24.2,100.4,55.8,defensive_event,c3323881-9112-4b7d-b3cf-06062fcb5902,Block,Granit Xhaka,Bayer Leverkusen,95.0,110.0,100.4,55.8,100.4,55.8,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,1,7.0,Granit Xhaka,Saved,0.020734,drawing,0,11,11,0,True
1,3895052,1,RB Leipzig,Bayer Leverkusen,8,9,From Keeper,Regular Play,2.28,137.0,35.1,18.3,84.9,61.7,attacking_event,d684e5e4-ee5a-4a7a-8013-d787ae77e932,Miscontrol,Xaver Schlager,RB Leipzig,117.0,137.0,35.1,18.3,35.1,18.3,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN,drawing,0,11,11,0,True
2,3895052,1,Bayer Leverkusen,RB Leipzig,9,10,Regular Play,Regular Play,2.45,147.0,97.9,67.1,22.1,12.9,defensive_event,5bc87cbd-db36-4bd5-a4d7-c07ed5435238,Clearance,Daniel Olmo Carvajal,RB Leipzig,130.0,146.0,22.1,12.9,22.1,12.9,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,True,Head,NaN,<NA>,<NA>,<NA>,True,NaN,0,NaN,NaN,NaN,NaN,drawing,0,11,11,0,True
3,3895052,1,RB Leipzig,Bayer Leverkusen,11,12,From Throw In,Regular Play,3.13,188.0,19.0,11.8,101.0,68.2,defensive_event,bbf1e609-60cb-4dcb-bd23-9ac34721002d,Duel,Jonas Hofmann,Bayer Leverkusen,150.0,181.0,101.0,68.2,101.0,68.2,NaN,NaN,NaN,Tackle,Success In Play,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,<NA>,True,NaN,0,NaN,NaN,NaN,NaN,drawing,0,11,11,0,True
4,3895052,1,Bayer Leverkusen,RB Leipzig,12,13,Regular Play,Regular Play,4.13,248.0,108.1,55.6,11.9,24.4,attacking_event,5bc6657b-a17c-486e-ad59-22672d97b128,Ball Receipt*,Jeremie Frimpong,Bayer Leverkusen,219.0,248.0,108.1,55.6,108.1,55.6,NaN,NaN,Incomplete,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN,drawing,0,11,11,0,True


## 7. Variables del área de acción inmediata

Para cada pérdida calculamos, en un radio configurable alrededor del punto de pérdida, indicadores de superioridad local que la literatura de *football analytics* asocia con el riesgo de transición peligrosa.

Las posiciones de los jugadores provienen de los datos **StatsBomb 360** cargados en la sección 2 (`freeze_lookup`). statsbombpy los expone a través de `sb.frames()`, que devuelve una fila por jugador y usa `id` como clave de evento (equivale al `id` de los eventos). En la sección 2 reagrupamos por `id` para reconstruir la lista de jugadores por evento. Las coordenadas se normalizan al mismo sistema que `loss_x`/`loss_y` (el equipo que pierde siempre ataca hacia *x* alto; su portería está en (0, 40)).

> **Disponibilidad de datos 360**: disponibles para Bundesliga 2023/24 y otras competiciones del open data. Si para alguna competición no existiese el fichero, la sección 2 asigna `freeze_lookup = {}` y las variables de zona quedarán como `NaN`.

Variables calculadas:

-   `zone_losing_players` / `zone_transition_players`: número de jugadores de cada equipo dentro del radio.
-   `zone_player_diff`: `losing − transition` (positivo = el equipo que pierde tiene más defensores en la zona).
-   `nearest_defender_dist_m`: distancia en metros del defensor más cercano al poseedor del balón.
-   `defenders_goalside_in_zone`: defensores de la zona cuya distancia al centro de la portería propia es menor que la del poseedor del balón.

In [10]:
# ── Parámetros del área de acción (modificar aquí) ────────────────────────
RADIUS_METERS = 10.0        # Radio en metros

# Dimensiones reales del campo y escala StatsBomb (campo estándar 105 m × 68 m)
PITCH_LENGTH_M = 105.0
PITCH_WIDTH_M  =  68.0
SB_LENGTH      = 120.0
SB_WIDTH       =  80.0

# Portería propia en el sistema normalizado del equipo que pierde
OWN_GOAL_X = 0.0
OWN_GOAL_Y = 40.0


def dist_m(x1, y1, x2, y2):
    """Distancia euclidea en metros entre dos puntos en coordenadas StatsBomb."""
    dx = (x2 - x1) * (PITCH_LENGTH_M / SB_LENGTH)
    dy = (y2 - y1) * (PITCH_WIDTH_M  / SB_WIDTH)
    return float(np.sqrt(dx**2 + dy**2))


_NAN_ZONE = {
    "zone_losing_players":        np.nan,
    "zone_transition_players":    np.nan,
    "zone_player_diff":           np.nan,
    "nearest_defender_dist_m":    np.nan,
    "defenders_goalside_in_zone": np.nan,
}


def zone_variables(row):
    """Calcula las 5 variables del área de acción inmediata para una pérdida."""
    frame = freeze_lookup.get(row.get("turnover_event_id"))
    if not frame:
        return _NAN_ZONE.copy()

    ball_x      = row["loss_x"]
    ball_y      = row["loss_y"]
    source      = row["turnover_event_source"]
    need_mirror = source in {"defensive_event", "next_possession_recovery"}

    players = []
    for p in frame:
        loc = p.get("location", [np.nan, np.nan])
        px, py   = float(loc[0]), float(loc[1])
        if need_mirror:
            px, py = mirror_x(px), mirror_y(py)

        teammate = bool(p.get("teammate", False))
        actor    = bool(p.get("actor",    False))

        # Asignación de equipo en el sistema normalizado del losing_team.
        # Para defensive_event/next_possession_recovery el actor es transition_team;
        # para attacking_event el actor es losing_team.
        if source in {"defensive_event", "next_possession_recovery"}:
            is_losing = not (teammate or actor)
        else:
            is_losing = teammate or actor

        players.append({
            "x": px, "y": py,
            "is_losing":    is_losing,
            "dist_to_ball": dist_m(px, py, ball_x, ball_y),
        })

    in_zone         = [p for p in players if p["dist_to_ball"] <= RADIUS_METERS]
    zone_losing     = sum(1 for p in in_zone if     p["is_losing"])
    zone_transition = sum(1 for p in in_zone if not p["is_losing"])

    losing_in_zone = [p for p in in_zone if p["is_losing"]]
    nearest_dist   = min((p["dist_to_ball"] for p in losing_in_zone), default=np.nan)

    d_ball_goal    = dist_m(ball_x, ball_y, OWN_GOAL_X, OWN_GOAL_Y)
    goalside_count = sum(
        1 for p in losing_in_zone
        if dist_m(p["x"], p["y"], OWN_GOAL_X, OWN_GOAL_Y) < d_ball_goal
    )

    return {
        "zone_losing_players":        zone_losing,
        "zone_transition_players":    zone_transition,
        "zone_player_diff":           zone_losing - zone_transition,
        "nearest_defender_dist_m":    nearest_dist,
        "defenders_goalside_in_zone": goalside_count,
    }


zone_df       = model_dataset.apply(zone_variables, axis=1, result_type="expand")
model_dataset = pd.concat(
    [model_dataset.reset_index(drop=True), zone_df.reset_index(drop=True)], axis=1
)

cobertura = zone_df["zone_losing_players"].notna().sum()
print(f"Pérdidas con datos 360: {cobertura:,} / {len(model_dataset):,}")
model_dataset[[
    "loss_x", "loss_y",
    "zone_losing_players", "zone_transition_players", "zone_player_diff",
    "nearest_defender_dist_m", "defenders_goalside_in_zone",
]].head(10)


Pérdidas con datos 360: 0 / 70


,loss_x,loss_y,zone_losing_players,zone_transition_players,zone_player_diff,nearest_defender_dist_m,defenders_goalside_in_zone
0,19.6,24.2,NaN,NaN,NaN,NaN,NaN
1,35.1,18.3,NaN,NaN,NaN,NaN,NaN
2,97.9,67.1,NaN,NaN,NaN,NaN,NaN
3,19.0,11.8,NaN,NaN,NaN,NaN,NaN
4,108.1,55.6,NaN,NaN,NaN,NaN,NaN
5,63.5,16.3,NaN,NaN,NaN,NaN,NaN
6,116.9,56.3,NaN,NaN,NaN,NaN,NaN
7,89.5,52.8,NaN,NaN,NaN,NaN,NaN
8,54.1,16.6,NaN,NaN,NaN,NaN,NaN
9,51.8,21.6,NaN,NaN,NaN,NaN,NaN


## 8. Variables de la última línea defensiva

La **vigilancia ofensiva** mide en qué medida la última línea defensiva puede controlar el espacio más peligroso en el momento de la pérdida: el rectángulo inmediatamente delante de los últimos defensores, desde el que un atacante puede progresar hacia portería sin encontrar oposición.

### Definición de la zona

Partiendo del fotograma 360, localizamos al **último defensor de campo** del equipo que pierde (el jugador con menor *x* en el sistema normalizado, excluyendo el portero). Su coordenada *x* marca el borde interior de la zona. Desde ahí, la zona se extiende `LAST_LINE_DEPTH_METERS` metros en dirección *x* creciente (hacia el campo del equipo atacante), con todo el ancho del campo en *y*.

### Variables calculadas

**a) `last_def_dist_to_goal_m`** Distancia en metros del último defensor de campo al centro de la portería propia (0, 40). Mide la profundidad de la línea: cuanto mayor, más espacio hay a explotar a sus espaldas.

**b) `last_line_losing_players` / `last_line_transition_players`** Número de jugadores de cada equipo dentro de la zona de última línea. El portero se excluye del conteo defensor; todos los atacantes visibles se incluyen.

**c) `last_line_player_diff`** `losing − transition`. Positivo: el equipo que defiende la transición tiene más jugadores en la zona. Negativo: está en inferioridad numérica en ese espacio crítico.

**d.1) `last_line_def_area_ratio` — Control territorial (Voronoi)** Se construye un diagrama de Voronoi usando **todos** los jugadores visibles del fotograma (no solo los de la zona). Usar todos los jugadores evita los artefactos de borde que ocurren cuando hay pocos jugadores en la zona: las celdas se extienden correctamente hasta los límites del campo gracias al *reflection trick* (cada punto se refleja en las cuatro líneas del campo antes de calcular Voronoi, técnica popularizada por Dan Nichol). Las celdas resultantes se recortan al rectángulo de la última línea con `shapely`. El ratio *área defensores / área total* expresa qué fracción del espacio crítico controla el equipo que defiende: `1.0` = control total, `0.0` = espacio completamente libre para el rival.

**d.2) `last_line_avg_marking_dist_m` — Presión de marcaje** Para cada atacante dentro de la zona, se calcula la distancia al defensor visible más cercano (buscando en todo el campo, no solo en la zona). El promedio de estas distancias captura si los atacantes en la zona están bien marcados o tienen libertad de movimiento. Complementa el ratio Voronoi: puede haber equilibrio territorial pero con atacantes alejados de sus marcadores, lo que aumenta el peligro real de la transición.

In [11]:
# ── Parámetros de la última línea (modificar aquí) ────────────────────────
LAST_LINE_DEPTH_METERS = 10.0   # Profundidad de la zona en metros

LAST_LINE_DEPTH_SB = LAST_LINE_DEPTH_METERS * (SB_LENGTH / PITCH_LENGTH_M)

from scipy.spatial import Voronoi as _Voronoi
from shapely.geometry import Polygon as _Polygon
from shapely.errors import GEOSException


def _reflect_2d(pts):
    """Reflection trick: refleja puntos en los 4 bordes del campo para Voronoi."""
    x0, x1, y0, y1 = 0.0, SB_LENGTH, 0.0, SB_WIDTH
    return np.vstack([
        pts,
        np.column_stack([2*x0 - pts[:, 0], pts[:, 1]]),
        np.column_stack([2*x1 - pts[:, 0], pts[:, 1]]),
        np.column_stack([pts[:, 0], 2*y0 - pts[:, 1]]),
        np.column_stack([pts[:, 0], 2*y1 - pts[:, 1]]),
    ])


def _voronoi_area_ratio(players, zone_poly):
    """
    Fracción del área de zone_poly controlada por el equipo que pierde (losing_team).
    Tesela con todos los jugadores visibles para evitar artefactos de borde.
    """
    if len(players) < 2:
        return np.nan

    coords    = np.array([[p["x"], p["y"]] for p in players])
    is_losing = np.array([p["is_losing"] for p in players])
    n         = len(players)

    try:
        vor = _Voronoi(_reflect_2d(coords))
    except Exception:
        return np.nan

    def_area   = 0.0
    total_area = 0.0

    for i in range(n):
        region = vor.regions[vor.point_region[i]]
        if -1 in region or len(region) < 3:
            continue
        try:
            poly    = _Polygon(vor.vertices[region])
            clipped = poly.intersection(zone_poly)
            if not clipped.is_valid or clipped.is_empty:
                continue
            area = clipped.area
            if is_losing[i]:
                def_area += area
            total_area += area
        except (GEOSException, Exception):
            continue

    return def_area / total_area if total_area > 0 else np.nan


_NAN_LL = {
    "last_def_dist_to_goal_m":      np.nan,
    "last_line_losing_players":     np.nan,
    "last_line_transition_players": np.nan,
    "last_line_player_diff":        np.nan,
    "last_line_def_area_ratio":     np.nan,
    "last_line_avg_marking_dist_m": np.nan,
}


def last_line_variables(row):
    """Calcula las 6 variables de la última línea defensiva para una pérdida."""
    frame = freeze_lookup.get(row.get("turnover_event_id"))
    if not frame:
        return _NAN_LL.copy()

    source      = row["turnover_event_source"]
    need_mirror = source in {"defensive_event", "next_possession_recovery"}

    players = []
    for p in frame:
        loc = p.get("location", [np.nan, np.nan])
        px, py = float(loc[0]), float(loc[1])
        if need_mirror:
            px, py = mirror_x(px), mirror_y(py)
        if not (np.isfinite(px) and np.isfinite(py)):
            continue

        teammate = bool(p.get("teammate", False))
        actor    = bool(p.get("actor",    False))
        keeper   = bool(p.get("keeper",   False))

        if source in {"defensive_event", "next_possession_recovery"}:
            is_losing = not (teammate or actor)
        else:
            is_losing = teammate or actor

        players.append({
            "x": px, "y": py,
            "is_losing": is_losing,
            "keeper":    keeper,
        })

    # Último defensor de campo (excluir portero)
    outfield_defenders = [p for p in players if p["is_losing"] and not p["keeper"]]
    if not outfield_defenders:
        return _NAN_LL.copy()

    last_def   = min(outfield_defenders, key=lambda p: p["x"])
    last_def_x = last_def["x"]

    # a) Distancia último defensor → portería propia
    last_def_dist_goal = dist_m(last_def["x"], last_def["y"], OWN_GOAL_X, OWN_GOAL_Y)

    # Zona: [last_def_x, last_def_x + LAST_LINE_DEPTH_SB] × [0, SB_WIDTH]
    zone_x_max = min(last_def_x + LAST_LINE_DEPTH_SB, SB_LENGTH)
    zone_poly  = _Polygon([
        (last_def_x, 0),
        (zone_x_max, 0),
        (zone_x_max, SB_WIDTH),
        (last_def_x, SB_WIDTH),
    ])

    # b) Jugadores en zona (porteros excluidos del lado defensor)
    in_zone_losing     = [
        p for p in players
        if p["is_losing"] and not p["keeper"]
        and last_def_x <= p["x"] <= zone_x_max
    ]
    in_zone_transition = [
        p for p in players
        if not p["is_losing"]
        and last_def_x <= p["x"] <= zone_x_max
    ]

    # d.1) Voronoi con todos los jugadores (excl. porteros), clipeado a zona
    players_for_vor = [p for p in players if not p["keeper"]]
    def_area_ratio  = _voronoi_area_ratio(players_for_vor, zone_poly)

    # d.2) Distancia media de marcaje para atacantes en zona
    if in_zone_transition:
        defenders_all = [p for p in players if p["is_losing"] and not p["keeper"]]
        if defenders_all:
            marking_dists = [
                min(dist_m(att["x"], att["y"], d["x"], d["y"]) for d in defenders_all)
                for att in in_zone_transition
            ]
            avg_marking = float(np.mean(marking_dists))
        else:
            avg_marking = np.nan
    else:
        avg_marking = np.nan

    return {
        "last_def_dist_to_goal_m":      last_def_dist_goal,
        "last_line_losing_players":     len(in_zone_losing),
        "last_line_transition_players": len(in_zone_transition),
        "last_line_player_diff":        len(in_zone_losing) - len(in_zone_transition),
        "last_line_def_area_ratio":     def_area_ratio,
        "last_line_avg_marking_dist_m": avg_marking,
    }


ll_df         = model_dataset.apply(last_line_variables, axis=1, result_type="expand")
model_dataset = pd.concat(
    [model_dataset.reset_index(drop=True), ll_df.reset_index(drop=True)], axis=1
)

cobertura_ll = ll_df["last_def_dist_to_goal_m"].notna().sum()
print(f"Pérdidas con datos de última línea: {cobertura_ll:,} / {len(model_dataset):,}")
model_dataset[[
    "loss_x", "loss_y",
    "last_def_dist_to_goal_m",
    "last_line_losing_players", "last_line_transition_players", "last_line_player_diff",
    "last_line_def_area_ratio", "last_line_avg_marking_dist_m",
]].head(10)


Pérdidas con datos de última línea: 0 / 70


,loss_x,loss_y,last_def_dist_to_goal_m,last_line_losing_players,last_line_transition_players,last_line_player_diff,last_line_def_area_ratio,last_line_avg_marking_dist_m
0,19.6,24.2,NaN,NaN,NaN,NaN,NaN,NaN
1,35.1,18.3,NaN,NaN,NaN,NaN,NaN,NaN
2,97.9,67.1,NaN,NaN,NaN,NaN,NaN,NaN
3,19.0,11.8,NaN,NaN,NaN,NaN,NaN,NaN
4,108.1,55.6,NaN,NaN,NaN,NaN,NaN,NaN
5,63.5,16.3,NaN,NaN,NaN,NaN,NaN,NaN
6,116.9,56.3,NaN,NaN,NaN,NaN,NaN,NaN
7,89.5,52.8,NaN,NaN,NaN,NaN,NaN,NaN
8,54.1,16.6,NaN,NaN,NaN,NaN,NaN,NaN
9,51.8,21.6,NaN,NaN,NaN,NaN,NaN,NaN


## 9. Dataframes finales

`model_dataset` contiene todas las perdidas en juego y la variable objetivo `transition_shot_within_window`.

Desde esa base generamos dos dataframes:

- `random_forest_dataset`: mas ancho, con mas variables candidatas y categoricas para que el bosque pueda seleccionar.
- `logistic_regression_dataset`: mas compacto, con menos columnas y menos duplicidad entre variables.

`dangerous_losses_window` contiene solo los positivos: perdidas que terminan en tiro rival dentro de la ventana configurable.

In [12]:
TARGET_COL = "transition_shot_within_window"

# Dataset amplio para Random Forest.
# Excluimos variables de fuga como seconds_to_shot, shot_outcome o xG:
# esas solo se conocen despues de saber que hubo tiro.
random_forest_cols = [
    TARGET_COL,
    "action_minute",
    "losing_team_score_state", "losing_team_score_diff",
    "transition_team_player_diff",
    "losing_team_players", "transition_team_players",
    "loss_x", "loss_y",
    "turnover_action_type", "turnover_event_source",
    "lost_possession_play_pattern", "transition_play_pattern",
    "under_pressure", "counterpress",
    "losing_team", "transition_team",
    "zone_losing_players", "zone_transition_players", "zone_player_diff",
    "nearest_defender_dist_m", "defenders_goalside_in_zone",
    "last_def_dist_to_goal_m",
    "last_line_losing_players", "last_line_transition_players", "last_line_player_diff",
    "last_line_def_area_ratio", "last_line_avg_marking_dist_m",
]

# Dataset compacto para regresion logistica.
# En numerica nos quedamos con transition_team_player_diff, no con los conteos duplicados.
# En posicion nos quedamos con loss_x/loss_y, no con transition_x/transition_y.
logistic_regression_cols = [
    TARGET_COL,
    "action_minute",
    "losing_team_score_diff",
    "transition_team_player_diff",
    "loss_x", "loss_y",
    "turnover_action_type", "turnover_event_source",
    "under_pressure", "counterpress",
    "zone_player_diff", "nearest_defender_dist_m", "defenders_goalside_in_zone",
    "last_def_dist_to_goal_m", "last_line_player_diff",
    "last_line_def_area_ratio", "last_line_avg_marking_dist_m",
]

review_cols = [
    "match_id", "period", "lost_possession", "transition_possession",
    "losing_team", "transition_team",
    "action_minute", "loss_x", "loss_y", "transition_x", "transition_y",
    "turnover_action_type", "turnover_event_source",
    TARGET_COL, "seconds_to_shot", "shot_player", "shot_outcome", "shot_statsbomb_xg",
]

random_forest_dataset = model_dataset[[c for c in random_forest_cols if c in model_dataset.columns]].copy()
logistic_regression_dataset = model_dataset[[c for c in logistic_regression_cols if c in model_dataset.columns]].copy()

dangerous_losses_window = (
    model_dataset[model_dataset[TARGET_COL] == 1]
    [[c for c in review_cols if c in model_dataset.columns]]
    .copy()
    .reset_index(drop=True)
)

print(f"Filas base: {len(model_dataset):,}")
print(f"Columnas Random Forest: {len(random_forest_dataset.columns)}")
print(f"Columnas regresion logistica: {len(logistic_regression_dataset.columns)}")
print(f"Positivos con tiro rival en <= {SHOT_WINDOW_SECONDS}s: {len(dangerous_losses_window):,}")

display(random_forest_dataset.head())
display(logistic_regression_dataset.head())
dangerous_losses_window


Filas base: 70
Columnas Random Forest: 28
Columnas regresion logistica: 17
Positivos con tiro rival en <= 10s: 3


,transition_shot_within_window,action_minute,losing_team_score_state,losing_team_score_diff,transition_team_player_diff,losing_team_players,transition_team_players,loss_x,loss_y,turnover_action_type,turnover_event_source,lost_possession_play_pattern,transition_play_pattern,under_pressure,counterpress,losing_team,transition_team,zone_losing_players,zone_transition_players,zone_player_diff,nearest_defender_dist_m,defenders_goalside_in_zone,last_def_dist_to_goal_m,last_line_losing_players,last_line_transition_players,last_line_player_diff,last_line_def_area_ratio,last_line_avg_marking_dist_m
0,1,1.87,drawing,0,0,11,11,19.6,24.2,Block,defensive_event,From Throw In,Regular Play,NaN,NaN,RB Leipzig,Bayer Leverkusen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,2.28,drawing,0,0,11,11,35.1,18.3,Miscontrol,attacking_event,From Keeper,Regular Play,NaN,NaN,RB Leipzig,Bayer Leverkusen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,2.45,drawing,0,0,11,11,97.9,67.1,Clearance,defensive_event,Regular Play,Regular Play,True,NaN,Bayer Leverkusen,RB Leipzig,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,3.13,drawing,0,0,11,11,19.0,11.8,Duel,defensive_event,From Throw In,Regular Play,True,NaN,RB Leipzig,Bayer Leverkusen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,4.13,drawing,0,0,11,11,108.1,55.6,Ball Receipt*,attacking_event,Regular Play,Regular Play,NaN,NaN,Bayer Leverkusen,RB Leipzig,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,transition_shot_within_window,action_minute,losing_team_score_diff,transition_team_player_diff,loss_x,loss_y,turnover_action_type,turnover_event_source,under_pressure,counterpress,zone_player_diff,nearest_defender_dist_m,defenders_goalside_in_zone,last_def_dist_to_goal_m,last_line_player_diff,last_line_def_area_ratio,last_line_avg_marking_dist_m
0,1,1.87,0,0,19.6,24.2,Block,defensive_event,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,2.28,0,0,35.1,18.3,Miscontrol,attacking_event,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,2.45,0,0,97.9,67.1,Clearance,defensive_event,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,3.13,0,0,19.0,11.8,Duel,defensive_event,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,4.13,0,0,108.1,55.6,Ball Receipt*,attacking_event,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,match_id,period,lost_possession,transition_possession,losing_team,transition_team,action_minute,loss_x,loss_y,transition_x,transition_y,turnover_action_type,turnover_event_source,transition_shot_within_window,seconds_to_shot,shot_player,shot_outcome,shot_statsbomb_xg
0,3895052,1,5,6,RB Leipzig,Bayer Leverkusen,1.87,19.6,24.2,100.4,55.8,Block,defensive_event,1,7.0,Granit Xhaka,Saved,0.020734
1,3895052,1,43,44,RB Leipzig,Bayer Leverkusen,22.92,47.2,11.6,72.8,68.4,Block,defensive_event,1,9.0,Jeremie Frimpong,Goal,0.491543
2,3895052,2,115,116,RB Leipzig,Bayer Leverkusen,56.45,67.9,11.4,52.1,68.6,Dispossessed,attacking_event,1,7.0,Victor Okoh Boniface,Off T,0.042780


## 10. Comprobaciones rapidas

In [ ]:
print("Tipos de perdida detectados")
display(
    model_dataset
    .groupby(["turnover_action_type", "turnover_event_source"], dropna=False)
    .agg(perdidas=("match_id", "count"), tiros_en_ventana=(TARGET_COL, "sum"))
    .assign(pct_tiro_en_ventana=lambda d: (100 * d["tiros_en_ventana"] / d["perdidas"]).round(1))
    .reset_index()
    .sort_values("perdidas", ascending=False)
)

print(f"Resumen por estado de marcador y diferencia numerica; tiro rival en <= {SHOT_WINDOW_SECONDS}s")
display(
    model_dataset
    .groupby(["losing_team_score_state", "transition_team_player_diff"], dropna=False)
    .agg(perdidas=("match_id", "count"), tiros_en_ventana=(TARGET_COL, "sum"))
    .assign(pct_tiro_en_ventana=lambda d: (100 * d["tiros_en_ventana"] / d["perdidas"]).round(1))
    .reset_index()
    .sort_values(["losing_team_score_state", "transition_team_player_diff"])
)
